In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from pathlib import PurePath, Path

from src.pyannote_person_detector import diarize_voice_rec
from src.tools.audio_files import get_audio_infos

from pprint import pp

# Check Meta Infos of Your Audio Files

## Trump and Zelensky Meeting

In [ ]:
path = 'path_here'

pp(get_audio_infos(path), width=1000)

## Trump Speak Single

In [ ]:
path = 'path_here'

pp(get_audio_infos(path), width=100)

## Zelensky Speak Single

In [ ]:
path = 'path_here'

pp(get_audio_infos(path), width=100)

# Diarize a File
## Initiate

In [ ]:
diar_obj = diarize_voice_rec(input_dir_to_diar='path_here',
                  input_file_name_to_diar='file_name_here',
                  )

## Diarize

In [ ]:
diar_obj.diarize(cuda_switch=True,
                 dump_switch=True)

## Split

In [ ]:
diar_obj.split()

### Save Split Informations
Saves a JSON which contains the speaker names and the start and end time for the splits.

In [ ]:
# default path is the default output path
diar_obj.split_dict_to_json()

# Compare to a Reference

In [ ]:
diar_obj.ref_cdist(ref_dir_path='path_here',
                   ref_file_name='file_name_here',
                   cuda_switch=True)

In [ ]:
df_distance = pd.DataFrame(diar_obj.dist_dict).transpose()

df_distance.set_index(df_distance.index.rename(['event', 'speaker_id', 'split_file', 'ref_file']), inplace=True)

print(df_distance.shape)

df_distance

### Add Informations

#### Start and End of a Split in the main file

In [ ]:
pd.set_option('display.max_rows', 500)
df_splits = pd.DataFrame(dict(sorted(diar_obj.split_dict.items()))).transpose()

df_splits.set_index(df_splits.index.rename(['event', 'speaker_id', 'split_file']), inplace=True)

df_splits.index = pd.MultiIndex.from_tuples([(PurePath(i[0]).stem,) + i[1:] for i in df_splits.index])

df_splits.set_index(df_splits.index.rename(['event', 'speaker_id', 'split_file']), inplace=True)

print(df_splits.shape)

df_splits

#### Join the DataFrames

In [ ]:
# Get ref_file as column from the index
df_distance_join= df_distance.reset_index('ref_file')

print(df_distance_join.shape)

df_distance_join

In [ ]:
df_all = df_distance_join.join(df_splits, how='inner')

print(df_all.shape)

df_all

### Examine for a Gap

**Now look at the values of the metric! Normally you see a strong drop or a gap at a certain point.  
This is an indication that all values less than or equal to this point belong to the speaker you are looking for.  
We have found that this value is usually around 800. However, this depends heavily on the quality of the audio   
files used. The cleaner they are (in terms of the speaker's pronunciation, audio quality, etc.), the better the result.**

**In this case, the gap is between just over 900.**

In [ ]:
df_all['metric_euclidean'].hist()

### Selection of Splits and Saving

In [ ]:
print(df_all.loc[df_all['metric_euclidean']<=960].shape)

df_all.loc[df_all['metric_euclidean']<=960]

In [ ]:
diar_obj.copy_sel_slices(dir_file_lst=list(df_all.loc[df_all['metric_euclidean']<=960].index),
                         speaker_name='dir_name')